In [13]:
import subprocess, sys, torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"Python: {sys.version}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "miditok>=3.0.0",
    "ncps>=0.0.7",
    "safetensors>=0.4.0",
    "pretty_midi>=0.2.10",
    "numpy>=1.24.0",
    "tqdm>=4.65.0",
], check=False)

# Pre-built Mamba wheels for Kaggle (PyTorch 2.9, CUDA 12.x, Python 3.12)
# Update these URLs if Kaggle updates its PyTorch/CUDA version
CAUSAL_CONV1D = (
    "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1/"
    "causal_conv1d-1.6.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)
MAMBA_SSM = (
    "https://github.com/state-spaces/mamba/releases/download/v2.3.1/"
    "mamba_ssm-2.3.1+cu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
)

if torch.cuda.is_available():
    print("\nInstalling Mamba from pre-built wheels...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        CAUSAL_CONV1D], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        MAMBA_SSM], capture_output=True)
    if r2.returncode != 0:
        print(f"Mamba wheel failed: {r2.stderr.decode()[-300:]}")
        print("GRU fallback will be used")
    else:
        print("Mamba installed successfully")
else:
    print("No GPU detected - GRU fallback will be used")

# Verify mamba
try:
    import mamba_ssm
    print(f"mamba-ssm {mamba_ssm.__version__} active")
except ImportError:
    print("mamba-ssm not available - GRU fallback active")

# Find and add project to Python path
import os
from pathlib import Path

def find_and_add_project():
    marker = 'piano_kaggle_session.py'
    for root, dirs, files in os.walk('/kaggle/input'):
        if marker in files:
            project_root = root
            if project_root not in sys.path:
                sys.path.insert(0, project_root)
            print(f"Project root: {project_root}")
            return project_root
    # Show what's available to help debug
    print("Could not find project. Available in /kaggle/input:")
    for item in Path('/kaggle/input').iterdir():
        print(f"  {item}")
        for subitem in item.iterdir():
            print(f"    {subitem.name}")
    raise FileNotFoundError("piano_kaggle_session.py not found in /kaggle/input")

PROJECT_ROOT = find_and_add_project()
print("Setup complete.")


PyTorch: 2.9.0+cu126
CUDA: 12.6
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
GPU: Tesla P100-PCIE-16GB

Installing Mamba from pre-built wheels...
Mamba installed successfully
mamba-ssm 2.3.1 active
Project root: /kaggle/input/datasets/chickaboomcmurtrie/magic-midi
Setup complete.


In [14]:
# ============================================
# EDIT THIS CELL TO CHANGE TRAINING SETTINGS
# ============================================
SCALE = "nano"        # Options: nano, micro, small, medium
MAX_EPOCHS = 5000     # Ceiling - training stops at plateau before this
# ============================================

from scale_config import list_presets, get_preset
list_presets()
print(f"\nSelected: {SCALE}")
get_preset(SCALE)


Scale     Params    Description
------    ------    -----------
nano      ~3M       ~3M params - 30min Colab session, pipeline validation
micro     ~8M       ~8M params - meaningful learning, ~2 epochs per 30min session
small     ~22M      ~22M params - full architecture, ~1 epoch per 30min session
medium    ~60M      ~60M params - serious model, needs Colab Pro or Kaggle

Selected: nano
[nano] ~3M params - 30min Colab session, pipeline validation


{'description': '~3M params - 30min Colab session, pipeline validation',
 'params': '~3M',
 'model': ModelConfig(vocab_size=2000, d_model=192, n_layers=4, d_state=16, d_conv=4, expand=2, cfc_units=192, cfc_backbone_units=96, cfc_backbone_layers=2, cfc_every_n_layers=2, num_attention_heads=4, attention_every_n_layers=2, attention_dropout=0.1, use_relative_attention=True, max_relative_distance=128, dropout=0.1, use_cfc=True, use_mamba=True, max_sequence_length=1024, debug=False),
 'train': TrainConfig(batch_size=4, grad_accumulation_steps=4, learning_rate=0.0005, weight_decay=0.01, max_epochs=50, warmup_steps=200, max_grad_norm=1.0, save_every_n_epochs=1, keep_every_n_epochs=10, checkpoint_dir='checkpoints/', use_wandb=False, seed=42, device='auto'),
 'data': DataConfig(maestro_path='maestro-v3.0.0', tokenizer_path='tokenizer.json', processed_path='processed/', vocab_size=2000, seed_length=128, continuation_length=256, stride=128, min_piece_length=1200, max_sequence_length=512, max_piece

In [17]:
from piano_kaggle_session import run_kaggle_session
run_kaggle_session(scale=SCALE, max_epochs=MAX_EPOCHS)


Project root found at: /kaggle/input/datasets/chickaboomcmurtrie/magic-midi
MAESTRO found at: /kaggle/input/datasets/chickaboomcmurtrie/maestro-v3-0-0/maestro-v3.0.0
Kaggle environment:
  Project root:   /kaggle/input/datasets/chickaboomcmurtrie/magic-midi
  MAESTRO root:   /kaggle/input/datasets/chickaboomcmurtrie/maestro-v3-0-0/maestro-v3.0.0
  Working dir:    /kaggle/working
  Checkpoints:    /kaggle/working/checkpoints
[nano] ~3M params - 30min Colab session, pipeline validation
Tokenizer/processed cache missing in /kaggle/working. Running preprocessing...
Tokenizer not found. Training REMI+BPE tokenizer...



Saved tokenizer to /kaggle/working/tokenizer/tokenizer.json
Loaded MAESTRO metadata: 1276 path entries from /kaggle/input/datasets/chickaboomcmurtrie/maestro-v3-0-0/maestro-v3.0.0/maestro-v3.0.0.csv


Tokenizing MAESTRO: 100%|██████████| 1276/1276 [02:17<00:00,  9.28it/s]



Preprocessing summary
  Total MIDI files scanned: 1276
  Total pieces kept: 1273
  Total tokens: 14954363
  Piece length mean/min/max: 11747.3/1338/52362
  Vocabulary coverage: 91.35% (1827/2000)
  Manifest saved: /kaggle/working/processed/manifest.json
Dataset split by piece: train=1145, val=63, test=65


/kaggle/input/datasets/chickaboomcmurtrie/magic-midi/model/cfc_block.py:115: UserWarning: CfC mode 'pure_memory' is unavailable in this ncps version. Falling back to mode='pure'.
  warnings.warn(


  Piano MIDI Model - Kaggle Session
  Scale:             nano
  Resume status:     fresh start
  Epoch progress:    0 / 5000
  GPU:               Tesla P100-PCIE-16GB (15.9 GB)
  MAESTRO root:      /kaggle/input/datasets/chickaboomcmurtrie/maestro-v3-0-0/maestro-v3.0.0
  Checkpoint dir:    /kaggle/working/checkpoints
  Processed dir:     /kaggle/working/processed
  Tokenizer path:    /kaggle/working/tokenizer/tokenizer.json
Estimated training memory: 0.05 GB (GPU available: 15.89 GB)
[Watchdog] Heartbeat: epoch=0 step=0


/kaggle/input/datasets/chickaboomcmurtrie/magic-midi/training/trainer.py:367: UserWarning: train_n_epochs interrupted: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
. Attempting emergency save.
  warnings.warn(


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from piano_kaggle_session import calibrate_on_kaggle
calibrate_on_kaggle()


In [ ]:
from piano_kaggle_session import generate_from_best_checkpoint
generate_from_best_checkpoint(scale=SCALE)


## Downloading your trained model

After training completes, find your files in the Kaggle output panel:
- `checkpoints/best.safetensors` - best model by validation loss
- `checkpoints/latest.safetensors` - most recent epoch
- `generated/` - sample MIDI continuations generated during training

To use locally:
```bash
python tools/generate_sample.py \
  --checkpoint path/to/best.safetensors \
  --seed your_seed.mid \
  --output generated.mid \
  --scale nano
```

To continue training in a new session:
Re-run the notebook - it auto-resumes from the latest checkpoint
if one exists in /kaggle/working/checkpoints/.


In [ ]:
# Diagnostic cell - run if other cells fail
import os
from pathlib import Path

print("=== Kaggle Environment Diagnostics ===")
print(f"\nWorking dir contents:")
for f in Path('/kaggle/working').iterdir():
    print(f"  {f.name}")

print(f"\n/kaggle/input contents:")
for item in Path('/kaggle/input').iterdir():
    print(f"  {item.name}/")
    try:
        for subitem in list(item.iterdir())[:5]:
            print(f"    {subitem.name}")
    except:
        pass

print(f"\nPython path:")
import sys
for p in sys.path[:5]:
    print(f"  {p}")

print(f"\nInstalled relevant packages:")
import subprocess
result = subprocess.run(['pip', 'show', 'miditok', 'ncps', 'mamba-ssm',
    'safetensors'], capture_output=True, text=True)
print(result.stdout)
